# Module 9: Neural NLP for real messages

Build a training-only vocabulary and neural text classifier on human-labeled messages, with a strong lexical baseline.

Use the complete source folder and the documented isolated CPU framework environment. Run in order, retain named text/tables, save, restart, and run all. First source/model downloads need internet; valid caches support replay. No earlier notebook state or paid account is needed.

The public script calls the same implementation. Small-model mechanics and valid software execution do not establish task quality or accessible notebook-interface operation.


In [ ]:
from pathlib import Path
import sys
import json
import tempfile
import platform
import importlib.metadata
import numpy as np
import pandas as pd

# Find shared code from the complete checkout/archive, not copied notebook logic.
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "ml_ai_it"]
ROOT = next((p for p in candidates if (p / "src/practical_ml_ai").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the complete ml_ai_it source folder.")
sys.path.insert(0, str(ROOT / "src"))
from practical_ml_ai.course_runner import run_module

# A new directory allows clean replay without silently replacing frozen evidence.
(ROOT / "runs").mkdir(exist_ok=True)
parent = Path(tempfile.mkdtemp(prefix="module09-notebook-", dir=ROOT / "runs"))
run_dir = parent / "study"
print("Evidence will be saved to:", run_dir)


## Inspect the real source and boundaries

These are actual human-labeled SMS records. Exact normalized texts do not cross splits; reports use hashes rather than raw messages.

Normalized text hashes group duplicate messages; conflicting labels are excluded before a seeded stratified 1,200-message sample and 60/20/20 split. TextVectorization adapts on training only, with at most 1,500 tokens and length 64. A 16-dimensional embedding, masked pooling, and sigmoid output train on real labels. TF-IDF/logistic is fitted on the same training partition.

Before running, identify the observation unit, input information, and fitting/selection boundary. No generated dataset or fabricated labels substitute for the external source.


In [ ]:
from practical_ml_ai.continuation_data import sms_splits
split, source_manifest = sms_splits()
for name, part in split.items():
    # Display IDs/counts rather than raw messages that may contain personal data.
    print(name, "rows:", len(part), "positive labels:", int(part.label.sum()))
print("First training text IDs:", split["train"].text_id.head(3).tolist())
print("Sampling rule:", source_manifest["sampling"])


## Predict and inspect the mechanism

Vocabulary adaptation is fitting. Including final-test words in that step would let the holdout influence the representation. The label is a human spam/ham annotation, not a property inferred from a phone number alone. Class imbalance can make accuracy look good while recall is poor.

Tokens are discrete representations; embeddings are learned numeric vectors. Padding supports fixed-length batches and should not become a real word signal. Global average pooling summarizes token vectors. Unknown words and truncated messages limit what the model can use.


In [ ]:
# Whitespace counts are a readability diagnostic, not the fitted vocabulary.
word_counts = split["train"].text.str.split().str.len()
print("Training whitespace-word counts: min/median/max",
      int(word_counts.min()), float(word_counts.median()), int(word_counts.max()))
print("The actual neural vocabulary is adapted only inside the training workflow.")


## Guided construction before independent practice

A token ID is an index, not an amount. In this hand example 0 is padding, 1 and 2 index vectors [1,0] and [0,2]. The masked mean of [1,2,0] is [.5,1], not [1/3,2/3]. In the real model, training learns vectors; the hand table only explains the operation.

Predict the output before running. Then complete the module guided page's independent modification.


In [ ]:
import numpy as np
learning_ids = np.array([1,2,0])
learning_embedding = np.array([[0.,0.],[1.,0.],[0.,2.]])
learning_mask = learning_ids != 0
learning_pooled = learning_embedding[learning_ids][learning_mask].mean(axis=0)
print("vectors",learning_embedding[learning_ids],"pooled",learning_pooled)
assert np.allclose(learning_pooled,[.5,1.])

## Run the shared workflow

The script and this cell call one implementation. Training/validation choices and default compute budgets are fixed before the final report. A source hash mismatch or completed-directory error should be investigated, not bypassed.

The short neural run may miss positives; compare it honestly with the lexical baseline.


In [ ]:
# Actual execution, not a mocked result. Change optional budgets only as a declared development experiment.
result = run_module(9, run_dir, epochs=3)
print((run_dir / "summary.txt").read_text(encoding="utf-8"))


## Inspect the exact tables

A vocabulary file and nonzero weight updates confirm actual training, but do not guarantee usefulness. The same final rows support both methods. Public SMS may contain personal details; public access and a license do not remove the need for restraint. Report a baseline win rather than assuming neural NLP must be superior. In the first default run, the neural model predicted no spam: TP 0, FN 30, TN 210, FP 0. Its 87.5% accuracy would conceal zero positive recall and balanced accuracy 0.5. The lexical baseline recovered 6 of 30 positives. This is an explicit failure case for diagnosis, not a deployable filter.

The first rows below are a preview. Open the full CSV/HTML report to inspect all values. Tables preserve the evidence without requiring a sight-dependent chart or screenshot.


In [ ]:
table = pd.read_csv(run_dir / "learning_history.csv")
print("Evidence table: learning_history.csv")
print(table.head(15).to_string(index=False))
print("Complete report:", run_dir / "index.html")


## Verify one interpretation

Use the actual output to check a denominator, fitting boundary, update, or budget. A passing software check is separate from whether the method is useful.


In [ ]:
predictions = pd.read_csv(run_dir / "test_predictions.csv")
print("Actual positive labels:", int(predictions.actual.sum()))
print("Predicted positives:", int(predictions.predicted.sum()))
print("Precision/recall may be poor even if overall accuracy looks high.")
print("Training-only vocabulary:", result["vocabulary_fitted_on"])


## Record versions and save

Retain the source/protocol manifest, output directory, commands, versions, and your own interpretation. Complete the lesson's practice and reflection; optional depth is skippable. Restart-and-run-all makes hidden state visible, but repeated public final data are not fresh independent evaluation.


In [ ]:
versions = {"python": platform.python_version()}
for package in ["numpy","pandas","scikit-learn","tensorflow-cpu"]:
    versions[package] = importlib.metadata.version(package)
(run_dir / "environment.json").write_text(json.dumps(versions, indent=2) + "\n", encoding="utf-8")
print("Reproduction versions:", versions)
print("Your saved evidence:", run_dir)


## Independent practice

Audit the text-hash boundaries and vocabulary fitting scope. Compare neural and lexical baseline precision/recall and identify a limitation from length/vocabulary. Inspect raw messages locally only as needed; report IDs and aggregate findings rather than publishing personal contact strings.

Optional: Study a CNN/recurrent encoder, a larger vocabulary, or a length change with a predefined development protocol. Measure compute and errors. A different encoder is optional, not an entrance requirement.

Explain one limitation and a next action. If using GenAI, verify its suggestions against actual code/tables and follow local disclosure rules. Instructors can adapt the linked practice criteria while preserving learning goals and accessibility.
